# w9_concat_5fold.ipynb — 两阶段折级裁决(BYOL→swin,@4096/600ep×5)

User (2026-07-21): 固定分割上 swin-bw 的 tag .742/.736(全场最高,两臂复
现)必须折级硬化。每折用**本折** BYOL 塔(5fold-2 已齐)在其 zsbest 选点
epoch 的 ckpt 热启动,swin168 精调 600ep(30% 预算)。cv worker 已移植
--init-ckpt(_bw 后缀,阶段一选点归阶段一)。参照 = swin 从零折
(.702±.034/tag .712)、i2ce 折(.730±.036/tag .712)、byol 折(tag
.730)。判据: bw 折 tag ≥ .73 且检索 ≥ swin 从零 → "30% 预算 + tag 红
利"升论文级;tag 缩回 .712 → 归档为固定分割骰子。AUTO-STOPS。


In [ ]:
# constants
import os

REPO = "/workspace/stable-query-latent"
URL = "https://github.com/Nice9Tian/stable-query-latent.git"
DATA_SRC = "/workspace/fusion_cache_w9"
DATA_RAM = "/dev/shm/fusion_cache_w9"
OUT_DIR = "/workspace/w9_cv_out"

ARM = "wcle_swin168step84loop2i2ce_icetf"
BYOL = "wcle_byol_bytf"
CAP, EPOCHS, N_FOLDS = 4096, 600, 5
os.makedirs(OUT_DIR, exist_ok=True)
print(f"concat 5-fold: {ARM}@{CAP} {EPOCHS}ep, warm from per-fold {BYOL} zsbest ckpts")


In [ ]:
# FORCE-sync repo to origin/main.
import os, importlib.util
if not os.path.isdir(os.path.join(REPO, ".git")):
    !git clone {URL} {REPO}
%cd {REPO}
!git remote set-url origin {URL}
!git fetch origin main
!git reset --hard origin/main
!git rev-parse --short HEAD
for pkg in ("sklearn", "scipy"):
    if importlib.util.find_spec(pkg) is None:
        !pip -q install scikit-learn scipy
        break
import sys
if REPO not in sys.path:
    sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, "Pod"))
import w9_jobs as J
print("machinery loaded")


In [ ]:
# Stage the corpus into RAM (same file set as w9_a100.ipynb).
import shutil
from pathlib import Path
REQUIRED = ["games.npz", "wiki_eval.npz", "wscan_gal_rev.npz",
            "wscan_pool_rev.npy", "wscan_pool_rev_rid.npy", "wscan_pool_rev_len.npy",
            "ss_queries_rev.npz", "ss_queries_rev_S.npy",
            "wiki_clean_views.npz", "sp_raw_views.npz", "tag_labels.npz",
            "wiki_eval_split.json", "_tag_splitM.json"]
src = Path(DATA_SRC)
missing = [f for f in REQUIRED if not (src / f).exists()]
assert not missing, f"missing in {DATA_SRC}: {missing}"
dst = Path(DATA_RAM)
dst.mkdir(parents=True, exist_ok=True)
for f in REQUIRED:
    s, d = src / f, dst / f
    if not d.exists() or d.stat().st_size != s.stat().st_size:
        print(f"staging {f} ({s.stat().st_size/1e9:.2f} GB) ...", flush=True)
        shutil.copyfile(s, d)
DATA_DIR = str(dst)
print("corpus in RAM:", DATA_DIR)

In [ ]:
# Full pool: must be READY on the volume; stage onto fast local storage.
import os, time
from pathlib import Path
from Pod.h5_staging import parallel_copy

ready = Path(DATA_SRC) / "full_pool_READY"
assert ready.exists(), "full pool not READY -- run a campaign notebook's build cell once"
src_v = Path(DATA_SRC) / "full_pool_fp16.npy"
src_m = Path(DATA_SRC) / "full_pool_meta.npz"
need = src_v.stat().st_size + (5 << 30)

def _free(p):
    st = os.statvfs(p)
    return st.f_bavail * st.f_frsize

dest_dir = None
for cand in ("/dev/shm", "/root/data", "/root"):
    Path(cand).mkdir(parents=True, exist_ok=True)
    if _free(cand) > need:
        dest_dir = Path(cand)
        break
if dest_dir is None:
    print("WARNING: no local space -- workers will mmap the NETWORK VOLUME copy.")
    FULL_POOL_PATH = str(src_v)
else:
    dst_v = dest_dir / "full_pool_fp16.npy"
    if dst_v.exists() and dst_v.stat().st_size == src_v.stat().st_size:
        print("local full pool already staged:", dst_v)
    else:
        t0 = time.time()
        tmp = dst_v.with_name(dst_v.name + ".copying")
        print(f"staging {src_v.stat().st_size/2**30:.0f} GiB -> {dst_v} ...", flush=True)
        parallel_copy(src_v, tmp, workers=8)
        os.replace(tmp, dst_v)
        print(f"staged in {(time.time()-t0)/60:.1f} min", flush=True)
    import shutil
    shutil.copyfile(src_m, dest_dir / "full_pool_meta.npz")
    FULL_POOL_PATH = str(dst_v)
print("FULL_POOL_PATH =", FULL_POOL_PATH)


In [ ]:
# Resolve per-fold BYOL warm ckpts, run 5 bw folds (round-robin GPUs).
import json, os, subprocess, threading, time
from pathlib import Path

cdir = Path(OUT_DIR) / "claims"
logd = Path(OUT_DIR) / "logs"
logd.mkdir(parents=True, exist_ok=True)
cdir.mkdir(parents=True, exist_ok=True)
gpus = J.detect_gpus()

todo = []
for k in range(N_FOLDS):
    nm = f"w9cv_{ARM}_fold{k}_g{CAP}_bw"
    if (Path(OUT_DIR) / f"tower_{nm}_fp_ep{EPOCHS}.npz").exists():
        print(f"[skip] {nm} done"); continue
    zb = Path(OUT_DIR) / f"zsbest_w9cv_{BYOL}_fold{k}_g{CAP}_fp.json"
    assert zb.exists(), f"missing stage-1 selection {zb}"
    be = json.loads(zb.read_text())["best_ep"]
    ck = Path(OUT_DIR) / f"ckpt_w9cv_{BYOL}_fold{k}_g{CAP}_fp_ep{be}.pt"
    if not ck.exists():
        cands = sorted(Path(OUT_DIR).glob(f"ckpt_w9cv_{BYOL}_fold{k}_g{CAP}_fp_ep*.pt"),
                       key=lambda q: abs(int(q.stem.split("_ep")[-1]) - be))
        assert cands, f"no BYOL ckpts for fold{k}"
        ck = cands[0]
        print(f"[fold{k}] zsbest ep{be} ckpt missing -> nearest {ck.name}")
    todo.append((k, nm, str(ck)))
print(f"{len(todo)} fold(s) to run")
stop_evt = threading.Event()
threading.Thread(target=J._monitor, args=([logd], stop_evt), daemon=True).start()
fails = []

def run_one(g, k, nm, ck):
    ok = J.try_claim(cdir, nm)
    if not ok:
        print(f"[claim] {nm} held -- waiting 130s", flush=True)
        time.sleep(130)
        ok = J.try_claim(cdir, nm)
    if not ok:
        print(f"[claim] {nm} held elsewhere -- skipped", flush=True); return
    cmd = ["python", "-u", J.CV_WORKER, "--data-dir", DATA_DIR, "--out-dir",
           OUT_DIR, "--repo", REPO, "--arm", ARM, "--fold", str(k),
           "--n-folds", str(N_FOLDS), "--anchor-cap", str(CAP),
           "--epochs", str(EPOCHS), "--ckpt-every", "50",
           "--init-ckpt", ck,
           "--full-pool", "--full-pool-path", FULL_POOL_PATH,
           "--claim-file", str(cdir / f"{nm}.claim")]
    print(f"[gpu{g}] start {nm} (warm={Path(ck).name})", flush=True)
    t0 = time.time()
    with open(logd / f"{nm}.log", "w") as fh:
        pr = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT,
                            env=dict(os.environ, CUDA_VISIBLE_DEVICES=g,
                                     PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True"))
    if pr.returncode != 0:
        (cdir / f"{nm}.claim").unlink(missing_ok=True); fails.append(nm)
    print(f"[gpu{g}] {'ok' if pr.returncode == 0 else 'FAIL'} {nm} "
          f"[{(time.time() - t0) / 3600:.1f}h]", flush=True)

# one consumer thread PER GPU, folds eaten sequentially (a 4096 swin
# tower needs ~45G -- two per 80G card do NOT fit).
import queue as _q
jq = _q.Queue()
for job in todo:
    jq.put(job)

def gpu_consumer(g, delay):
    time.sleep(delay)
    while True:
        try:
            k, nm, ck = jq.get_nowait()
        except _q.Empty:
            return
        run_one(g, k, nm, ck)

ths = [threading.Thread(target=gpu_consumer, args=(g, 60 * i))
       for i, g in enumerate(gpus)]
for t in ths:
    t.start()
for t in ths:
    t.join()
stop_evt.set()
print(f"done; {len(fails)} failed")


In [ ]:
# Fold-level verdict: concat bw vs swin-scratch / i2ce / byol.
import json
import numpy as np
from pathlib import Path

def rows_of(arm, sfx=""):
    out = {}
    for k in range(N_FOLDS):
        p = Path(OUT_DIR) / f"zsbest_w9cv_{arm}_fold{k}_g{CAP}{sfx}_fp.json"
        if p.exists():
            out[k] = json.loads(p.read_text())
    return out

def show(lab, rows):
    if not rows:
        print(f"{lab:26s} (pending)"); return
    M = lambda f: np.mean([r[f] for r in rows.values()])
    S = lambda f: np.std([r[f] for r in rows.values()])
    print(f"{lab:26s} n={len(rows)} neu {M('nm_neutral'):.3f}±{S('nm_neutral'):.3f} "
          f"non {M('nm_noname'):.3f}±{S('nm_noname'):.3f} "
          f"tag {M('tag_neutral'):.3f}±{S('tag_neutral'):.3f}/{M('tag_noname'):.3f}±{S('tag_noname'):.3f}")

bw = rows_of(ARM, "_bw")
sc = rows_of(ARM)
ic = rows_of("wcle_i2ce_icetf")
by = rows_of("wcle_byol_bytf")
show("concat bw 600ep", bw)
show("swin scratch 2000ep", sc)
show("i2ce full 2000ep", ic)
show("byol (stage-1)", by)
for ref, lab in ((sc, "vs swin-scratch"), (ic, "vs i2ce")):
    ks = [k for k in bw if k in ref]
    if ks:
        for f in ("nm_noname", "tag_neutral", "tag_noname"):
            d = [bw[k][f] - ref[k][f] for k in ks]
            print(f"  {lab} Δ{f}: {[round(x,3) for x in d]} mean {np.mean(d):+.3f} "
                  f"wins {sum(1 for x in d if x>0)}/{len(d)}")


In [ ]:
# AUTO-STOP: stop THIS pod when the queue has finished (results live on the
# network volume; idle GPU time is pure waste). Uses the hardened ladder in
# VICReg_review/pod_selfstop.py. Set AUTO_STOP=False to keep the pod alive.
AUTO_STOP = True
if AUTO_STOP:
    import sys
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    from VICReg_review import pod_selfstop
    if fails:
        print(f"NOTE: {len(fails)} job(s) FAILED -- logs in {OUT_DIR}/logs; "
              "stopping anyway to avoid idle burn.")
    pod_id, api_key, ctl = pod_selfstop.preflight("")
    pod_selfstop.stop_pod(pod_id, api_key, ctl)
else:
    print("AUTO_STOP disabled -- remember to stop the pod yourself.")